In [1]:
### inyoung_correlationwithMGpipeline.ipynb
### This script is written to analyze the processed twop data using MG pipeline. 
### Input: processed twop data
### Output: spike raster plot and pairwise correlation plot
### Lines to change: folder_path (2), n_iterations (7) and subsample_ratio (8)

### JSY, 25/05/14

In [2]:
cd "C:\Users\jasmineyeo\Documents\GitHub\V1_SpatialModulation"

C:\Users\jasmineyeo\Documents\GitHub\V1_SpatialModulation


In [3]:

# Import required libraries
import os
import matplotlib.pyplot as plt
import numpy as np
import scipy.io as sio
import oasis.functions
from tqdm import tqdm
import oasis
import pandas as pd

def normalize_crosscorr(C):
    """
    Normalize a 2D cross-correlation matrix to range [-1, 1].
    
    Parameters:
        C (numpy.ndarray): Input 2D correlation matrix.
    Returns:
        numpy.ndarray: Normalized correlation matrix.
    """
    C_min = np.min(C)
    C_max = np.max(C)
    
    # Avoid division by zero if the matrix is constant
    if C_max == C_min:
        return np.zeros_like(C)
    
    C_norm = ((C - C_min) / (C_max - C_min)) * 2 - 1
    return C_norm

def calculate_cross_correlation_multiple_iterations(data, n_iterations=10, subsample_ratio=0.8, min_std_threshold=1e-10):
    """
    Calculate cross-correlation matrix across multiple iterations by subsampling the data.
    
    Parameters:
        data (numpy.ndarray): Neural activity data with shape (n_cells, n_frames)
        n_iterations (int): Number of iterations for correlation calculation
        subsample_ratio (float): Fraction of data to use in each iteration (0-1)
        min_std_threshold (float): Minimum standard deviation threshold to avoid division by zero
    
    Returns:
        numpy.ndarray: Mean cross-correlation matrix across iterations
        numpy.ndarray: Standard deviation of cross-correlations across iterations
    """
    n_cells, n_frames = data.shape
    all_correlation_matrices = np.zeros((n_iterations, n_cells, n_cells))
    
    # First, check for cells with no variation and report them
    std_per_cell = np.std(data, axis=1)
    zero_std_cells = np.where(std_per_cell < min_std_threshold)[0]
    if len(zero_std_cells) > 0:
        print(f"Warning: {len(zero_std_cells)} cells have near-zero standard deviation")
        if len(zero_std_cells) < 20:  # Only print all indices if there aren't too many
            print(f"Cell indices with near-zero std: {zero_std_cells}")
        else:
            print(f"First 20 cell indices with near-zero std: {zero_std_cells[:20]}...")
    
    for i in tqdm(range(n_iterations), desc="Calculating correlations"):
        # Randomly subsample frames
        n_subsample = int(n_frames * subsample_ratio)
        subsample_indices = np.random.choice(n_frames, n_subsample, replace=False)
        
        # Calculate correlation on subsampled data
        subsampled_data = data[:, subsample_indices]
        
        # Modified approach for correlation calculation
        # 1. Calculate means for each cell
        cell_means = np.mean(subsampled_data, axis=1, keepdims=True)
        
        # 2. Calculate standard deviations for each cell
        std_values = np.std(subsampled_data, axis=1)
        
        # 3. Replace zero standard deviations with a small value
        std_values[std_values < min_std_threshold] = min_std_threshold
        
        # 4. Calculate correlation matrix using numpy's corrcoef with handling for potential issues
        try:
            # Center the data (subtract mean)
            centered_data = subsampled_data - cell_means
            
            # Calculate correlation matrix
            # First calculate covariance
            cov_matrix = np.zeros((n_cells, n_cells))
            for i_cell in range(n_cells):
                for j_cell in range(n_cells):
                    cov_matrix[i_cell, j_cell] = np.mean(centered_data[i_cell] * centered_data[j_cell])
            
            # Then normalize by standard deviations to get correlation
            corr_matrix = np.zeros((n_cells, n_cells))
            for i_cell in range(n_cells):
                for j_cell in range(n_cells):
                    if i_cell == j_cell:
                        corr_matrix[i_cell, j_cell] = 1.0  # Diagonal is always 1
                    else:
                        corr = cov_matrix[i_cell, j_cell] / (std_values[i_cell] * std_values[j_cell])
                        # Clip to handle any numerical instability
                        corr_matrix[i_cell, j_cell] = np.clip(corr, -1.0, 1.0)
            
            all_correlation_matrices[i] = corr_matrix
            
        except Exception as e:
            print(f"Error in iteration {i}: {str(e)}")
            # Fill with identity matrix if calculation fails
            all_correlation_matrices[i] = np.eye(n_cells)
    
    # Calculate mean and std of correlation matrices
    mean_correlation = np.nanmean(all_correlation_matrices, axis=0)
    std_correlation = np.nanstd(all_correlation_matrices, axis=0)
    
    # Fill any remaining NaN values with zeros
    mean_correlation = np.nan_to_num(mean_correlation)
    std_correlation = np.nan_to_num(std_correlation)
    
    return mean_correlation, std_correlation

def check_spike_data(spikes, min_spikes_per_cell=5):
    """
    Check if spike data has enough signal for correlation analysis.
    
    Parameters:
        spikes (numpy.ndarray): Spike data with shape (n_cells, n_frames)
        min_spikes_per_cell (int): Minimum number of spikes required for a cell
    
    Returns:
        bool: True if data is suitable, False otherwise
    """
    n_cells, n_frames = spikes.shape
    
    # Count non-zero elements for each cell
    spike_counts = np.sum(spikes > 0, axis=1)
    
    # Check how many cells have enough spikes
    valid_cells = np.sum(spike_counts >= min_spikes_per_cell)
    
    # Print diagnostic information
    print(f"Total cells: {n_cells}")
    print(f"Cells with at least {min_spikes_per_cell} spikes: {valid_cells} ({valid_cells / n_cells * 100:.1f}%)")
    
    # Calculate some statistics about the spike data
    mean_spikes_per_cell = np.mean(spike_counts)
    median_spikes_per_cell = np.median(spike_counts)
    max_spikes_per_cell = np.max(spike_counts)
    
    print(f"Mean spikes per cell: {mean_spikes_per_cell:.2f}")
    print(f"Median spikes per cell: {median_spikes_per_cell:.2f}")
    print(f"Max spikes per cell: {max_spikes_per_cell:.2f}")
    
    # Check if at least 10% of cells have enough spikes
    return valid_cells / n_cells >= 0.1

def prepare_spikes_for_correlation(spikes, epsilon=1e-6):
    """
    Prepare spike data for correlation analysis by adding a small amount of noise
    to cells with no variation.
    
    Parameters:
        spikes (numpy.ndarray): Spike data with shape (n_cells, n_frames)
        epsilon (float): Small value to add to ensure non-zero std dev
    
    Returns:
        numpy.ndarray: Prepared spike data
    """
    spikes_modified = spikes.copy()
    
    # Find cells with no variation (constant values)
    cell_stds = np.std(spikes, axis=1)
    zero_std_cells = np.where(cell_stds < 1e-10)[0]
    
    if len(zero_std_cells) > 0:
        print(f"Adding small random noise to {len(zero_std_cells)} cells with no variation")
        
        # Add tiny random noise to these cells
        for cell in zero_std_cells:
            # Only add noise to a small percentage of frames to maintain sparsity
            noise_frames = np.random.choice(spikes.shape[1], size=max(int(spikes.shape[1] * 0.01), 10), replace=False)
            spikes_modified[cell, noise_frames] += epsilon * np.random.random(size=len(noise_frames))
    
    return spikes_modified

c:\Users\jasmineyeo\AppData\Local\anaconda3\envs\JSY_SpatialMod\Lib\site-packages\oasis\functions.py:13: UserWarning: Could not find cvxpy. Don't worry, you can still use OASIS, just not the slower interior point methods we compared to in the papers.
  warn("Could not find cvxpy. Don't worry, you can still use OASIS, " +


In [4]:
# Find the number of subfolders in the folder
folder_path = r'\\goard-nas1\Goard_Lab\Jasmine\toshare\inyoung\20250519_IYH\B1_cellcountcheck'
subfolders = [f.path for f in os.scandir(folder_path) if f.is_dir()]
num_subfolders = len(subfolders)

# Set parameters for correlation calculation
n_iterations = 100  # Number of iterations
subsample_ratio = 0.8  # Use 90% of frames in each iteration

# Loop through the subfolders
for subfolder in subfolders:
    try:
        # Get the base path and record name
        basepath = subfolder
        rec_name = os.path.basename(subfolder)
        print(f"Processing folder: {rec_name}, subfolder {subfolders.index(subfolder) + 1} of {num_subfolders}")
        
        # Find all .mat files in the subfolder
        mat_files = [f for f in os.listdir(basepath) if f.endswith('.mat')]
        
        if not mat_files:
            print(f"No .mat files found in {rec_name}")
            continue
            
        for mat_file in mat_files:
            try:
                mat_file_path = os.path.join(basepath, mat_file)
                print(f"Loading .mat file: {mat_file}")
                
                # Load the .mat file
                mat_data = sio.loadmat(mat_file_path)
                
                # Extract the 'data' field
                if 'data' not in mat_data:
                    print("No 'data' field found in this .mat file")
                    continue
                    
                data_struct = mat_data['data']
                print(f"Data type: {data_struct.dtype}")
                
                # Extract each field from the structure
                data_fields = {}
                for field_name in data_struct.dtype.names:
                    data_fields[field_name] = data_struct[0, 0][field_name]
                
                # Extract fields of interest
                filename = data_fields['filename']
                numFrames = data_fields['numFrames'][0][0] if data_fields['numFrames'].size > 0 else 0
                frameRate = data_fields['frameRate'][0][0] if data_fields['frameRate'].size > 0 else 0
                cellMasks = data_fields['cellMasks']
                DFF_final = data_fields['DFF']
                
                # Print summary information
                print(f"  Filename: {filename[0] if filename.size > 0 else 'Empty'}")
                print(f"  Number of frames: {numFrames}")
                print(f"  Frame rate: {frameRate}")
                n_cells = cellMasks.shape[1] if hasattr(cellMasks, 'shape') else 0
                print(f"  Number of cells detected: {n_cells}")
                
                if n_cells == 0 or numFrames == 0:
                    print("No cells or frames detected, skipping this file")
                    continue
                
                # Initialize arrays for spike data
                spikes = np.zeros([n_cells, numFrames])
                norm_spikes = np.zeros([n_cells, numFrames])
                
                # Calculate spikes for each cell
                print("Calculating spikes...")
                for c in tqdm(range(n_cells)):
                    try:
                        # Deconvolved spiking activity
                        g = oasis.functions.estimate_time_constant(DFF_final[c,:].copy(), 1)
                        _, spikes[c,:] = oasis.oasisAR1(DFF_final[c,:].copy(), g)
                        
                        # Normalize spikes
                        spikes_min = np.min(spikes[c])
                        spikes_max = np.max(spikes[c])
                        if spikes_max > spikes_min:  # Avoid division by zero
                            norm_spikes[c] = (spikes[c] - spikes_min) / (spikes_max - spikes_min)
                    except Exception as e:
                        print(f"Error processing cell {c}: {e}")
                
                # Save the processed data
                twop_dict = {
                    'filename': filename,
                    'numFrames': numFrames,
                    'frameRate': frameRate,
                    'cellMasks': cellMasks,
                    'DFF_final': DFF_final,
                    'spikes': spikes,
                    'norm_spikes': norm_spikes
                }
                
                # Plot spikes
                print("Plotting spike data...")
                fig, axs = plt.subplots(1, 2, figsize=(20, 6))
                
                im0 = axs[0].imshow(spikes, aspect='auto', cmap='hot', interpolation='nearest')
                axs[0].set_title(f'Spikes for {rec_name}')
                axs[0].set_xlabel('Frames')
                axs[0].set_ylabel('Cells')
                fig.colorbar(im0, ax=axs[0], label='Spike amplitude')
                
                im1 = axs[1].imshow(norm_spikes, aspect='auto', cmap='hot', interpolation='nearest')
                axs[1].set_title(f'Normalized spikes for {rec_name}')
                axs[1].set_xlabel('Frames')
                axs[1].set_ylabel('Cells')
                fig.colorbar(im1, ax=axs[1], label='Normalized spike amplitude')
                
                plt.tight_layout()
                spike_fig_path = os.path.join(basepath, f"{rec_name}_spikes.jpg")
                plt.savefig(spike_fig_path, dpi=300)
                plt.close()
                print(f"Saved spike plot to {spike_fig_path}")
                
                # Check spike data and prepare it if needed
                print("Analyzing spike statistics...")
                is_suitable = check_spike_data(spikes, min_spikes_per_cell=5)
                if is_suitable:
                    print("Spike data is suitable for correlation analysis")
                    spikes_prepared = prepare_spikes_for_correlation(spikes)
                else:
                    print("WARNING: Spike data has very few spikes. Adding minimal noise to enable correlation calculation.")
                    spikes_prepared = prepare_spikes_for_correlation(spikes, epsilon=1e-5)
                
                # Calculate cross-correlation with multiple iterations for DFF
                print(f"Calculating DFF cross-correlations across {n_iterations} iterations...")
                mean_corr_dff, std_corr_dff = calculate_cross_correlation_multiple_iterations(
                    DFF_final, n_iterations=n_iterations, subsample_ratio=subsample_ratio
                )
                
                # Calculate cross-correlation with multiple iterations for spikes
                print(f"Calculating spike cross-correlations across {n_iterations} iterations...")
                mean_corr_spikes, std_corr_spikes = calculate_cross_correlation_multiple_iterations(
                    spikes_prepared, n_iterations=n_iterations, subsample_ratio=subsample_ratio
                )
                
                # Normalize the mean correlation matrices
                norm_corr_dff = normalize_crosscorr(mean_corr_dff)
                norm_corr_spikes = normalize_crosscorr(mean_corr_spikes)
                
                # Before plotting, take absolute values of correlations:
                abs_corr_dff = np.abs(norm_corr_dff)
                abs_corr_spikes = np.abs(mean_corr_spikes)

                # Plot the cross-correlation matrices
                plt.figure(figsize=(20, 6))
                
                plt.subplot(1, 2, 1)
                im1 = plt.imshow(abs_corr_dff, aspect='auto', cmap='coolwarm', interpolation='nearest', vmin=0, vmax=1)
                plt.colorbar(im1, label='Cross-correlation (DFF)')
                plt.title(f'Mean Cross-correlation (DFF)\n{n_iterations} iterations')
                plt.xlabel('Cells')
                plt.ylabel('Cells')
                
                plt.subplot(1, 2, 2)
                im2 = plt.imshow(abs_corr_spikes, aspect='auto', cmap='coolwarm', interpolation='nearest', vmin=0, vmax=1)
                plt.colorbar(im2, label='Cross-correlation (Spikes)')
                plt.title(f'Mean Cross-correlation (Spikes)\n{n_iterations} iterations')
                plt.xlabel('Cells')
                plt.ylabel('Cells')
                
                # plt.subplot(1, 3, 3)
                # im3 = plt.imshow(std_corr_dff, aspect='auto', cmap='viridis', interpolation='nearest')
                # plt.colorbar(im3, label='Std Dev of Correlations')
                # plt.title(f'Std Dev of Cross-correlations\n{n_iterations} iterations')
                # plt.xlabel('Cells')
                # plt.ylabel('Cells')
                
                plt.tight_layout()
                corr_fig_path = os.path.join(basepath, f"{rec_name}_cross_correlation_{n_iterations}iter.jpg")
                plt.savefig(corr_fig_path, dpi=300)
                plt.close()
                print(f"Saved correlation plot to {corr_fig_path}")
                
                # Save the correlation matrices
                corr_dict = {
                    'mean_corr_dff': mean_corr_dff,
                    'std_corr_dff': std_corr_dff,
                    'mean_corr_spikes': mean_corr_spikes,
                    'std_corr_spikes': std_corr_spikes,
                    'n_iterations': n_iterations,
                    'subsample_ratio': subsample_ratio
                }
                
                npy_filename = os.path.splitext(mat_file)[0] + f'_correlations_{n_iterations}iter.npy'
                npy_file_path = os.path.join(basepath, npy_filename)
                np.save(npy_file_path, corr_dict)
                print(f"Saved correlation data to {npy_filename}")
                
                csv_filename = os.path.splitext(mat_file)[0] + f'_correlations_{n_iterations}iter.csv'
                csv_file_path = os.path.join(basepath, csv_filename)
                pd.DataFrame(mean_corr_dff).to_csv(csv_file_path, index=False)
                print(f"Saved correlation data to {csv_filename}")
                
            except Exception as e:
                print(f"Error processing file {mat_file}: {e}")
    
    except Exception as e:
        print(f"Error processing folder {subfolder}: {e}")

print("Processing complete!")

Processing folder: B1_Ctrl4_D90_org1-002, subfolder 1 of 32
Loading .mat file: B1_registered_data.mat
Data type: [('filename', 'O'), ('numFrames', 'O'), ('xPixels', 'O'), ('yPixels', 'O'), ('map_type', 'O'), ('frameRate', 'O'), ('avg_projection', 'O'), ('frame_F', 'O'), ('activity_map', 'O'), ('cellMasks', 'O'), ('raw_F', 'O'), ('normtype', 'O'), ('neuropil_F', 'O'), ('r_neuropil', 'O'), ('DFF_raw', 'O'), ('DFF', 'O')]
  Filename: B1_registered.tif
  Number of frames: 4507
  Frame rate: 15.023206948701938
  Number of cells detected: 95
Calculating spikes...


  0%|          | 0/95 [00:00<?, ?it/s]c:\Users\jasmineyeo\AppData\Local\anaconda3\envs\JSY_SpatialMod\Lib\site-packages\oasis\functions.py:817: FutureWarning: Beginning in SciPy 1.17, multidimensional input will be treated as a batch, not `ravel`ed. To preserve the existing behavior and silence this warning, `ravel` arguments before passing them to `toeplitz`.
  A = scipy.linalg.toeplitz(xc[np.arange(lags)],
C:\Users\jasmineyeo\AppData\Local\Temp\ipykernel_42532\2687361336.py:74: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  _, spikes[c,:] = oasis.oasisAR1(DFF_final[c,:].copy(), g)
100%|██████████| 95/95 [00:00<00:00, 1884.49it/s]

Plotting spike data...


Saved spike plot to \\goard-nas1\Goard_Lab\Jasmine\toshare\inyoung\20250519_IYH\B1_cellcountcheck\B1_Ctrl4_D90_org1-002\B1_Ctrl4_D90_org1-002_spikes.jpg
Analyzing spike statistics...
Total cells: 95
Cells with at least 5 spikes: 94 (98.9%)
Mean spikes per cell: 786.75
Median spikes per cell: 564.00
Max spikes per cell: 2393.00
Spike data is suitable for correlation analysis
Calculating DFF cross-correlations across 100 iterations...


Calculating correlations: 100%|██████████| 100/100 [00:07<00:00, 13.63it/s]


...

Saved spike plot to \\goard-nas1\Goard_Lab\Jasmine\toshare\inyoung\20250519_IYH\B1_cellcountcheck\#1_d150_Dup_org4-002\#1_d150_Dup_org4-002_spikes.jpg
Analyzing spike statistics...
Total cells: 292
Cells with at least 5 spikes: 291 (99.7%)
Mean spikes per cell: 552.02
Median spikes per cell: 292.50
Max spikes per cell: 2606.00
Spike data is suitable for correlation analysis
Calculating DFF cross-correlations across 100 iterations...


Calculating correlations: 100%|██████████| 100/100 [01:35<00:00,  1.04it/s]


Calculating spike cross-correlations across 100 iterations...


Calculating correlations: 100%|██████████| 100/100 [01:36<00:00,  1.03it/s]


Saved correlation plot to \\goard-nas1\Goard_Lab\Jasmine\toshare\inyoung\20250519_IYH\B1_cellcountcheck\#1_d150_Dup_org4-002\#1_d150_Dup_org4-002_cross_correlation_100iter.jpg
Saved correlation data to #1_data_correlations_100iter.npy
Saved correlation data to #1_data_correlations_100iter.csv
Processing complete!
